In [1]:
import os
import torch
from pymilvus import connections, Collection
from transformers import AutoModel, AutoTokenizer
import ollama
from typing import TypedDict
from langgraph.graph import StateGraph, END

# ========================
# CONFIGURAÇÃO DO MILVUS
# ========================
connections.connect("default", host="127.0.0.1", port="19530")
COLLECTION_NAME = "rag_embeddings_milvus"
collection = Collection(COLLECTION_NAME)

# ========================
# EMBEDDINGS
# ========================
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].numpy().tolist()

# ========================
# ESTADO COMPARTILHADO
# ========================
class ChatState(TypedDict):
    input: str
    query: str
    context: str
    refs: str
    images: str
    answer: str

# ========================
# AGENTE 1: RECUPERA TEXTO
# ========================
def retrieve_text_context(state: ChatState) -> ChatState:
    query = state["input"]
    query_emb = get_embedding(query)

    collection.load()
    results = collection.search(
        data=[query_emb],
        anns_field="embedding",
        param={"metric_type": "IP", "params": {"nprobe": 10}},
        limit=10,
        output_fields=["source_file", "source_url", "chunk_index", "chunk_text"]
    )

    contexts = []
    refs = []
    for r in results[0]:
        chunk_text = r.entity.get("chunk_text")
        source = r.entity.get("source_file")
        url = r.entity.get("source_url")
        contexts.append(chunk_text)
        refs.append(f"📄 {source} | 🔗 {url}")

    return {**state, "query": query, "context": "\n\n".join(contexts), "refs": "\n".join(refs)}

# ========================
# AGENTE 2: RECUPERA IMAGENS
# ========================
def retrieve_image_context(state: ChatState) -> ChatState:
    query = state["input"]
    query_emb = get_embedding(query)

    image_collection = Collection("image_descriptions")
    image_collection.load()

    results = image_collection.search(
        data=[query_emb],
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": 10}},
        limit=5,
        output_fields=["id", "url", "category", "titles", "texts"]
    )

    images = []
    for r in results[0]:
        url = r.entity.get("url")
        titles = r.entity.get("titles")
        texts = r.entity.get("texts")
        category = r.entity.get("category")
        score = r.distance
        images.append(
            f"🖼️ {category} | {titles} | {texts[:80]}... ({url}) [score={score:.3f}]"
        )

    return {**state, "images": "\n".join(images)}

# ========================
# AGENTE 3: GERA RESPOSTA
# ========================
def generate_answer(state: ChatState) -> ChatState:
    prompt = f"""
Você é um assistente técnico especializado em licenciamento ambiental (EIA/RIMA).
Responda à pergunta do usuário **usando apenas o contexto fornecido**.

Contexto textual:
{state['context']}

Contexto visual (descrições de imagens):
{state['images']}

Pergunta:
{state['query']}
    """
    response = ollama.chat(
        model="mistral:7b",
        messages=[
            {"role": "system", "content": "Você é um assistente técnico ambiental especializado em EIA/RIMA."},
            {"role": "user", "content": prompt}
        ]
    )
    return {**state, "answer": response["message"]["content"]}

# ========================
# CONSTRUÇÃO DO GRAFO
# ========================
builder = StateGraph(ChatState)

builder.add_node("text_retriever", retrieve_text_context)
builder.add_node("image_retriever", retrieve_image_context)
builder.add_node("answer_generator", generate_answer)

builder.set_entry_point("text_retriever")
builder.add_edge("text_retriever", "image_retriever")
builder.add_edge("image_retriever", "answer_generator")
builder.add_edge("answer_generator", END)

graph = builder.compile()

# ========================
# LOOP DE CHAT
# ========================
print("🤖 Chatbot Multiagente EIA/RIMA usando Milvus + Mistral 7B (Ollama). Digite 'sair' para encerrar.\n")

while True:
    user_input = input("Você: ")
    if user_input.lower() in ["sair", "exit", "quit"]:
        break

    state = {
        "input": user_input,
        "query": "",
        "context": "",
        "refs": "",
        "images": "",
        "answer": ""
    }

    result = graph.invoke(state)

    print("\nBot:", result["answer"])
    print("\n--- Fontes ---")
    print(result["refs"])
    if result["images"]:
        print("\n--- Imagens relacionadas ---")
        print(result["images"])


H:\GitR\Database-Chroma-RAG-Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🤖 Chatbot Multiagente EIA/RIMA usando Milvus + Mistral 7B (Ollama). Digite 'sair' para encerrar.



Você:  Durante o processo de estudos para duplicação, foi constatado que é necessário a retirada de vegetação nativa da mata atlântica na extensão da SP-97. Como se deve proceder para diminuir o impacto gerado devido a retirada da mesma segundo a regulação vigente?



Bot:  Para minimizar o impacto causado pela remoção de vegetação nativa da Mata Atlântica durante o processo de duplicação, é recomendável seguir uma abordagem integrada de manejo. Aqui estão algumas medidas que podem ser adotadas para a diminuição do impacto:

1. **Supressão Gradual e Unidirecional:** A supressão deve ser realizada de forma gradual, começando na borda do fragmento e avançando em direção ao seu interior, seguindo até o fragmento de vegetação nativa mais próximo.

2. **Programa de Acompanhamento de Supressão de Vegetação Nativa:** Necessário ter um plano para acompanhar a supressão para garantir que os procedimentos sejam efetuados de maneira adequada.

3. **Criação de Passagens de Fauna:** Essas estruturas irão conectar o fragmento que será suprimido aos fragmentos adjacentes dentro da propriedade, facilitando o deslocamento seguro da fauna e promovendo a conectividade ecológica entre os diferentes fragmentos florestais.

4. **Criação de Áreas para Abrigo e Afugentame

Você:  sair
